# 02. Gameplay Summary & Cleaning

**Goal**: Analyze player activity, filter out short sessions, and trim long sessions.

**Inputs**:
- `data/processed/merged_telemetry.csv` (Output from Step 1)

**Outputs**:
- `data/processed/2_cleaned_telemetry_for_modelling.csv` (Filtered content)
- `data/processed/2_player_summary.csv` (Stats)

In [2]:
# Analyis & Filtering
import pandas as pd
import os

# Inputs & Outputs
INPUT_PATH = 'data/processed/merged_telemetry.csv'
OUT_CLEANED = 'data/processed/2_cleaned_telemetry_for_modelling.csv'
OUT_SUMMARY = 'data/processed/2_player_summary.csv'

print("Loading processed telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows. Unique Users: {df['userId'].nunique()}")
    
    if 'userId' in df.columns:
        # 1. Calculate Per-Player Duration
        # Group keys: userId
        # Assuming sorted by user + time from Step 1, but we'll sort per user later just in case for trimming
        
        user_stats = df.groupby('userId').size().reset_index(name='total_rows')
        user_stats['duration_min'] = (user_stats['total_rows'] * 30) / 60
        
        print("\n--- Initial Stats ---")
        print(user_stats['duration_min'].describe())
        
        # 2. Define Filters
        min_duration = 20
        cap_duration = 45
        max_rows = int((cap_duration * 60) / 30) # 90 rows
        
        valid_users = user_stats[user_stats['duration_min'] >= min_duration]
        dropped_users = user_stats[user_stats['duration_min'] < min_duration]
        
        print(f"\nFiltering: Keeping {len(valid_users)} users (>= 20m). Dropping {len(dropped_users)} users (< 20m).")
        
        # 3. Apply Filtering & Trimming
        # We only keep rows for valid users
        df_filtered = df[df['userId'].isin(valid_users['userId'])].copy()
        
        # Sort properly to ensure we keep the FIRST 45 mins
        t_col = 'timestamp' if 'timestamp' in df_filtered.columns else 'time'
        # Ensure sorting column exists and is usable
        if t_col in df_filtered.columns:
             # Helper for sort if not already done
             df_filtered['sort_helper'] = pd.to_numeric(df_filtered[t_col], errors='coerce')
             if df_filtered['sort_helper'].isna().all():
                  df_filtered['sort_helper'] = pd.to_datetime(df_filtered[t_col], errors='coerce')
             df_filtered.sort_values(by=['userId', 'sort_helper'], inplace=True)
             df_filtered.drop(columns=['sort_helper'], inplace=True)
             
        # Trim > 45 mins (Keep first 90 rows per user)
        # GroupBy head(90)
        df_final = df_filtered.groupby('userId').head(max_rows)
        
        print(f"\nTrimming: Reduced rows from {len(df_filtered)} to {len(df_final)} (Cap at {cap_duration}m / {max_rows} rows).")
        
        # 4. Final Summary
        final_stats = df_final.groupby('userId').size().reset_index(name='Data points')
        final_stats['Duration (minutes)'] = (final_stats['Data points'] * 30) / 60
        final_stats.rename(columns={'userId': 'Player ID'}, inplace=True)
        final_stats = final_stats[['Player ID', 'Duration (minutes)', 'Data points']]
        final_stats.sort_values('Duration (minutes)', ascending=False, inplace=True)
        
        print("\n--- Final Cleaned Summary (Top 5) ---")
        print(final_stats.head(5).to_string(index=False))
        
        # 5. Export
        os.makedirs('data/processed', exist_ok=True)
        df_final.to_csv(OUT_CLEANED, index=False)
        final_stats.to_csv(OUT_SUMMARY, index=False)
        print(f"\nSaved cleaned dataset to {OUT_CLEANED}")
        print(f"Saved summary to {OUT_SUMMARY}")
        
    else:
        print("Error: 'userId' column missing in dataset.")
else:
    print(f"Error: File not found at {INPUT_PATH}. Please run 01_Data_Loading_and_Merging.ipynb first.")

Loading processed telemetry data...
Loaded 3408 rows. Unique Users: 57

--- Initial Stats ---
count     57.000000
mean      29.894737
std       23.428793
min        1.000000
25%       12.500000
50%       31.000000
75%       38.500000
max      136.000000
Name: duration_min, dtype: float64

Filtering: Keeping 38 users (>= 20m). Dropping 19 users (< 20m).

Trimming: Reduced rows from 3130 to 2715 (Cap at 45m / 90 rows).

--- Final Cleaned Summary (Top 5) ---
               Player ID  Duration (minutes)  Data points
6974892348d53c4152cf1421                45.0           90
6974acd8315a39e91bc1e52f                45.0           90
6974f3cc5d8e98b89f0a5d07                45.0           90
697502455d8e98b89f0a9f38                45.0           90
69757efd2890de71026a214a                45.0           90

Saved cleaned dataset to data/processed/2_cleaned_telemetry_for_modelling.csv
Saved summary to data/processed/2_player_summary.csv
